# 02 — Text Processing

This notebook demonstrates the text processing pipeline for NeuroForge.
After documents are ingested, raw text often contains noise from OCR, PDF extraction,
or format conversion. The `TextCleaner` removes this noise while preserving meaningful content.

---

## Text Cleaning

The `TextCleaner` applies a conservative multi-step pipeline:

1. **Remove headers/footers** — detect repeated lines across pages
2. **Remove page numbers** — standalone page number lines
3. **Fix OCR artifacts** — ligatures, smart quotes, common misreads
4. **Remove garbage chars** — control characters, null bytes
5. **Normalize whitespace** — collapse extra spaces/newlines
6. **Preserve formatting** — keep bullets and numbered lists intact

Design principle: **conservative** — better to leave slightly noisy text than to lose real content.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.processing.cleaning import TextCleaner

cleaner = TextCleaner()
print("TextCleaner loaded successfully.")

### Example 1: Noisy OCR Output

Simulating text extracted from a scanned PDF with typical OCR noise:

In [ ]:
noisy_text = """Company Report Header
\x00\x01The \ufb01rst \ufb01ndings of our research show signi\ufb01cant results.
The \ufb02ow of data was measured across    multiple     systems.

- 3 -

Key results:
- Item one: 42 patients enrolled
- Item two: 15 controls selected
\x00
Page 4

1. First step in the process
2. Second step follows





\u201cThis is quoted text,\u201d said the researcher.
The e\ufb00ect size was 0.45 \u2013 0.67.
~~~~ garbage ~~~~
Company Report Header"""

print("=== BEFORE CLEANING ===")
print(repr(noisy_text[:300]))
print("...")
print(f"\nLength: {len(noisy_text)} chars")

In [ ]:
cleaned_text = cleaner.clean(noisy_text)

print("=== AFTER CLEANING ===")
print(cleaned_text)
print(f"\nLength: {len(cleaned_text)} chars (was {len(noisy_text)})")

### Example 2: Clean Text (Should Pass Through)

Already-clean text should remain unchanged:

In [ ]:
clean_text = """Introduction

This document describes the architecture of our system.
The following components are included:

- Authentication module
- Data processing pipeline
- Storage layer

Steps to deploy:

1. Configure environment variables
2. Run database migrations
3. Start the application server

For more details, see the appendix."""

result = cleaner.clean(clean_text)

print("=== CLEAN INPUT ===")
print(clean_text)
print("\n=== AFTER CLEANING ===")
print(result)
print(f"\nChanged: {clean_text.strip() != result}")

### Example 3: Step-by-Step Pipeline

Running each cleaning step individually to see its effect:

In [ ]:
sample = "The \ufb01le contains \x00null\x00 bytes and   extra   spaces.\n\n\n\n\nToo many newlines.\n- 5 -\n"

steps = [
    ("Original", sample),
    ("After remove_headers_footers", cleaner.remove_headers_footers(sample)),
    ("After remove_page_numbers", cleaner.remove_page_numbers(sample)),
    ("After fix_ocr_artifacts", cleaner.fix_ocr_artifacts(sample)),
    ("After remove_garbage_chars", cleaner.remove_garbage_chars(sample)),
    ("After normalize_whitespace", cleaner.normalize_whitespace(sample)),
    ("After preserve_formatting", cleaner.preserve_formatting(sample)),
]

for name, text in steps:
    print(f"--- {name} ---")
    print(repr(text))
    print()

### Example 4: Header/Footer Detection

Demonstrating detection and removal of repeated headers/footers across pages:

In [ ]:
# Simulate a multi-page document
pages = []
for i in range(5):
    pages.append(
        f"ACME Corp - Internal Report\n"
        f"This is the actual content of page {i+1}.\n"
        f"It contains important information.\n"
        f"Confidential - Do Not Distribute"
    )

multi_page_text = "\f".join(pages)

print("=== BEFORE ===")
print(multi_page_text[:200])
print("...")

result = cleaner.remove_headers_footers(multi_page_text)
print("\n=== AFTER ===")
print(result[:200])
print(f"\nHeader 'ACME Corp' removed: {'ACME Corp' not in result}")
print(f"Footer 'Confidential' removed: {'Confidential' not in result}")
print(f"Content preserved: {'actual content' in result}")

---

**Text Cleaning complete.** The `TextCleaner` is ready for use in the document processing pipeline.

Import it in downstream notebooks with:
```python
from src.processing.cleaning import TextCleaner
```

---

## Intelligent Chunking

After cleaning, documents need to be split into smaller **chunks** for embedding and retrieval.
The `DocumentChunker` supports three strategies:

| Strategy | Description | Best For |
|----------|-------------|----------|
| `section` | Split at heading boundaries, then token-chunk within sections | Structured docs (papers, reports) |
| `paragraph` | Split on double-newline paragraph boundaries | Long prose (essays, articles) |
| `token` | Fixed-size sliding window with overlap | Unstructured text (transcripts) |

Each chunk receives:
- A deterministic **chunk ID** (`{source_hash}_{index:04d}`)
- **Metadata**: section heading, token count, character positions
- Correct **ordering** via `chunk_index`

In [ ]:
from src.processing.chunking import DocumentChunker
from models import Document, DocumentMetadata, InputFormat, Section

chunker = DocumentChunker(max_tokens=500, overlap=50)
print(f"DocumentChunker ready (max_tokens={chunker.max_tokens}, overlap={chunker.overlap})")

### Sample Document

Let's create a structured markdown document to demonstrate each strategy:

In [ ]:
sample_content = """# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that focuses on building systems
that learn from data. Rather than being explicitly programmed, these systems improve
their performance through experience.

## Supervised Learning

In supervised learning, models are trained on labeled data. The algorithm learns a
mapping from inputs to outputs. Common algorithms include linear regression, decision
trees, and neural networks. The key challenge is generalization — performing well on
unseen data rather than just memorizing the training set.

## Unsupervised Learning

Unsupervised learning finds patterns in unlabeled data. Clustering algorithms like
K-means group similar data points together. Dimensionality reduction techniques like
PCA find lower-dimensional representations. These methods are useful for exploratory
data analysis and feature engineering.

## Reinforcement Learning

Reinforcement learning trains agents through rewards and penalties. The agent learns
a policy that maximizes cumulative reward over time. Applications include game playing,
robotics, and recommendation systems. Key concepts include the exploration-exploitation
tradeoff and the Markov decision process framework."""

sample_doc = Document(
    content=sample_content,
    metadata=DocumentMetadata(source="ml_textbook_ch1.md", format=InputFormat.MARKDOWN),
)

print(f"Document: {sample_doc.metadata.source}")
print(f"Content length: {len(sample_content)} chars")

### Strategy 1: Section-based Chunking (default)

Splits at heading boundaries first. Each section becomes one or more chunks:

In [ ]:
section_chunks = chunker.chunk(sample_doc, strategy="section")

print(f"Section strategy produced {len(section_chunks)} chunks:\n")
for chunk in section_chunks:
    print(f"  [{chunk.id}] index={chunk.chunk_index}")
    print(f"    Section: {chunk.metadata.section_heading}")
    print(f"    Tokens: {chunk.metadata.token_count}")
    print(f"    Preview: {chunk.content[:60]}...")
    print()

### Strategy 2: Paragraph-based Chunking

Splits on double newlines — good for flowing prose:

In [ ]:
para_chunks = chunker.chunk(sample_doc, strategy="paragraph")

print(f"Paragraph strategy produced {len(para_chunks)} chunks:\n")
for chunk in para_chunks:
    print(f"  [{chunk.id}] index={chunk.chunk_index}")
    print(f"    Tokens: {chunk.metadata.token_count}")
    print(f"    Chars: [{chunk.metadata.start_char}:{chunk.metadata.end_char}]")
    print(f"    Preview: {chunk.content[:60]}...")
    print()

### Strategy 3: Token-based Chunking

Fixed-size sliding window with overlap — works for any text:

In [ ]:
# Use a smaller window to demonstrate multiple chunks on this short text
small_chunker = DocumentChunker(max_tokens=80, overlap=15)
token_chunks = small_chunker.chunk(sample_doc, strategy="token")

print(f"Token strategy (max=80, overlap=15) produced {len(token_chunks)} chunks:\n")
for chunk in token_chunks:
    print(f"  [{chunk.id}] index={chunk.chunk_index}, tokens={chunk.metadata.token_count}")
    print(f"    Preview: {chunk.content[:50]}...")
    print()

### Comparing Strategies

Let's compare all three strategies on the same document:

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(sample_content))

strategies = ["section", "paragraph", "token"]
comparison_chunker = DocumentChunker(max_tokens=100, overlap=20)

print(f"Document: {sample_doc.metadata.source}")
print(f"Total tokens: {total_tokens}")
print(f"Chunker settings: max_tokens=100, overlap=20")
print("=" * 60)

for strategy in strategies:
    chunks = comparison_chunker.chunk(sample_doc, strategy=strategy)
    token_counts = [c.metadata.token_count for c in chunks]
    avg_tokens = sum(token_counts) / len(token_counts) if token_counts else 0
    
    print(f"\n{strategy.upper()} strategy:")
    print(f"  Chunks produced: {len(chunks)}")
    print(f"  Avg tokens/chunk: {avg_tokens:.1f}")
    print(f"  Min tokens: {min(token_counts)}")
    print(f"  Max tokens: {max(token_counts)}")
    print(f"  Headings preserved: {sum(1 for c in chunks if c.metadata.section_heading)}")

### Chunk IDs and Ordering

Chunk IDs are deterministic — the same document always produces the same IDs:

In [ ]:
import hashlib

source_hash = hashlib.sha256(sample_doc.metadata.source.encode()).hexdigest()[:8]
print(f"Source: {sample_doc.metadata.source}")
print(f"Hash prefix: {source_hash}")
print(f"\nChunk IDs (section strategy):")

chunks = chunker.chunk(sample_doc, strategy="section")
for chunk in chunks:
    assert chunk.id == f"{source_hash}_{chunk.chunk_index:04d}"
    assert chunk.chunk_index == chunks.index(chunk)  # ordering preserved
    print(f"  {chunk.id} (index={chunk.chunk_index})")

print("\n✓ All chunk IDs match expected format")
print("✓ Ordering is sequential and zero-indexed")

---

**Intelligent Chunking complete.** The `DocumentChunker` is ready for use in the pipeline.

Import it with:
```python
from src.processing.chunking import DocumentChunker
```

Next up: structure extraction to detect tables, lists, code blocks, and section hierarchy.

---

## Structure Extraction

The `StructureExtractor` detects document structure using regex and pattern matching:

- **Section tree** — builds a heading hierarchy (h1 > h2 > h3)
- **Tables** — detects pipe-delimited and whitespace-aligned tables
- **Lists** — detects bullet and numbered lists with nesting
- **Code blocks** — detects fenced code blocks (``` delimited)

Structural metadata is then attached to chunks so retrieval can filter by element type.

In [ ]:
from src.processing.structure import StructureExtractor, DocumentStructure

extractor = StructureExtractor()
print("StructureExtractor loaded successfully.")

### Example: Rich Document Structure

Let's analyze a document that contains headings, a table, a list, and a code block:

In [ ]:
structured_content = """# Data Science Guide

An overview of tools and techniques.

## Libraries

Key libraries for data science:

| Library | Purpose | Language |
|---------|---------|----------|
| pandas | Data manipulation | Python |
| numpy | Numerical computing | Python |
| scikit-learn | Machine learning | Python |

## Setup Steps

Follow these steps:

- Install Python 3.11+
- Create a virtual environment
  - Use venv or conda
  - Activate it
- Install dependencies

### Example Code

```python
import pandas as pd
import numpy as np

df = pd.DataFrame({"a": [1, 2, 3]})
print(df.describe())
```

# Conclusion

That covers the basics.
"""

structured_doc = Document(
    content=structured_content,
    metadata=DocumentMetadata(source="ds_guide.md", format=InputFormat.MARKDOWN),
)

structure = extractor.extract(structured_doc)
print(f"Document: {structured_doc.metadata.source}")
print(f"Content length: {len(structured_content)} chars")

### Section Tree

The section tree shows the heading hierarchy:

In [ ]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent
        print(f"{prefix}{'#' * node.level} {node.heading} (chars {node.content_start}-{node.content_end})")
        if node.children:
            print_tree(node.children, indent + 1)

print("Section Tree:")
print_tree(structure.sections)

### Detected Tables

In [ ]:
print(f"Tables found: {len(structure.tables)}")
for i, table in enumerate(structure.tables):
    print(f"  Table {i+1}: {table.rows} rows x {table.columns} columns")
    print(f"    Position: chars {table.start_char}-{table.end_char}")
    print(f"    Preview: {structured_content[table.start_char:table.start_char+60]}...")

### Detected Lists

In [ ]:
print(f"Lists found: {len(structure.lists)}")
for i, lst in enumerate(structure.lists):
    print(f"  List {i+1}: {lst.items} items, nesting depth={lst.nesting_depth}")
    print(f"    Position: chars {lst.start_char}-{lst.end_char}")
    print(f"    Preview: {structured_content[lst.start_char:lst.start_char+60]}...")

### Detected Code Blocks

In [ ]:
print(f"Code blocks found: {len(structure.code_blocks)}")
for i, cb in enumerate(structure.code_blocks):
    print(f"  Block {i+1}: language={cb.language}")
    print(f"    Position: chars {cb.start_char}-{cb.end_char}")
    print(f"    Content preview: {cb.content[:60]}...")

### Annotating Chunks with Structure

After chunking, we can annotate each chunk with its structural context:

In [ ]:
from src.processing.chunking import DocumentChunker

chunker = DocumentChunker(max_tokens=200, overlap=30)
chunks = chunker.chunk(structured_doc, strategy="section")

annotated_chunks = extractor.annotate_chunks(chunks, structure)

print(f"Annotated {len(annotated_chunks)} chunks:\n")
for chunk in annotated_chunks:
    info = getattr(chunk.metadata, 'structure_info', None) or chunk.metadata.__dict__.get('structure_info')
    print(f"  Chunk {chunk.chunk_index}: {chunk.metadata.section_heading}")
    if info:
        for key, val in info.items():
            print(f"    {key}: {val}")
    else:
        print("    (no structural elements)")
    print()

---

**Structure Extraction complete.** The `StructureExtractor` is ready for use in the pipeline.

Import it with:
```python
from src.processing.structure import StructureExtractor, DocumentStructure
```

Next up: knowledge extraction using LLM-based analysis.